# Updated for Google Colab
This notebook should be run in Colab. Upload the CSV and adjust the file path if needed.

Q1. Spark Initialization and Data Loading

In [2]:
import pyspark
print(f"PySpark version: {pyspark.__version__}")

PySpark version: 4.1.2


In [3]:
import os
os.environ['PYSPARK_PYTHON'] = 'python'
os.environ['PYSPARK_DRIVER_PYTHON'] = 'python'

from pyspark.sql import SparkSession

In [4]:
spark = SparkSession.builder.appName('Bank Fraud')\
    .config('spark.python.worker.faulthandler.enabled', 'true')\
    .config('spark.python.worker.reuse', 'false')\
    .getOrCreate()

In [5]:
df = spark.read.format("csv").option("header","True").option("InferSchema","True").load("Bank_Transaction_Fraud_Detection.csv")

In [6]:
df.printSchema()

root
 |-- Customer_ID: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- State: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Bank_Branch: string (nullable = true)
 |-- Account_Type: string (nullable = true)
 |-- Transaction_ID: string (nullable = true)
 |-- Transaction_Date: string (nullable = true)
 |-- Transaction_Time: timestamp (nullable = true)
 |-- Transaction_Amount: double (nullable = true)
 |-- Merchant_ID: string (nullable = true)
 |-- Transaction_Type: string (nullable = true)
 |-- Merchant_Category: string (nullable = true)
 |-- Account_Balance: double (nullable = true)
 |-- Transaction_Device: string (nullable = true)
 |-- Transaction_Location: string (nullable = true)
 |-- Device_Type: string (nullable = true)
 |-- Is_Fraud: integer (nullable = true)
 |-- Transaction_Currency: string (nullable = true)
 |-- Customer_Contact: string (nullable = true)
 |-- 

In [7]:
df.show(5,truncate=False)

+------------------------------------+-------------------+------+---+-----------+------------------+-------------------------+------------+------------------------------------+----------------+-------------------+------------------+------------------------------------+----------------+-----------------+---------------+------------------+--------------------------+-----------+--------+--------------------+----------------+-----------------------+-----------------------+
|Customer_ID                         |Customer_Name      |Gender|Age|State      |City              |Bank_Branch              |Account_Type|Transaction_ID                      |Transaction_Date|Transaction_Time   |Transaction_Amount|Merchant_ID                         |Transaction_Type|Merchant_Category|Account_Balance|Transaction_Device|Transaction_Location      |Device_Type|Is_Fraud|Transaction_Currency|Customer_Contact|Transaction_Description|Customer_Email         |
+------------------------------------+--------------

Q2. RDD Implementation

In [8]:
import pyspark
import sys

print("PySpark:", pyspark.__version__)
print("Spark:", spark.version)
print("Python:", sys.version)

PySpark: 4.1.2
Spark: 4.1.2
Python: 3.13.1 (tags/v3.13.1:0671451, Dec  3 2024, 19:06:28) [MSC v.1942 64 bit (AMD64)]


In [9]:
import sys
import os

print(sys.executable)
print(os.path.exists(sys.executable))

c:\Program Files\Python313\python.exe
True


In [10]:
# Convert DataFrame into RDD
transaction_rdd = df.rdd

print("RDD Created Successfully")
print(transaction_rdd)

RDD Created Successfully
MapPartitionsRDD[18] at javaToPython at NativeMethodAccessorImpl.java:0


In [11]:
total_transactions = transaction_rdd.count()

print("Total Transactions :", total_transactions)

Total Transactions : 200000


In [12]:
transaction_rdd.take(5)

[Row(Customer_ID='d5f6ec07-d69e-4f47-b9b4-7c58ff17c19e', Customer_Name='Osha Tella', Gender='Male', Age=60, State='Kerala', City='Thiruvananthapuram', Bank_Branch='Thiruvananthapuram Branch', Account_Type='Savings', Transaction_ID='4fa3208f-9e23-42dc-b330-844829d0c12c', Transaction_Date='23-01-2025', Transaction_Time=datetime.datetime(2026, 7, 3, 16, 4, 7), Transaction_Amount=32415.45, Merchant_ID='214e03c5-5c34-40d1-a66c-f440aa2bbd02', Transaction_Type='Transfer', Merchant_Category='Restaurant', Account_Balance=74557.27, Transaction_Device='Voice Assistant', Transaction_Location='Thiruvananthapuram, Kerala', Device_Type='POS', Is_Fraud=0, Transaction_Currency='INR', Customer_Contact='+9198579XXXXXX', Transaction_Description='Bitcoin transaction', Customer_Email='oshaXXXXX@XXXXX.com'),
 Row(Customer_ID='7c14ad51-781a-4db9-b7bd-67439c175262', Customer_Name='Hredhaan Khosla', Gender='Female', Age=51, State='Maharashtra', City='Nashik', Bank_Branch='Nashik Branch', Account_Type='Business'

In [13]:
for row in transaction_rdd.take(5):
    print(row)

Row(Customer_ID='d5f6ec07-d69e-4f47-b9b4-7c58ff17c19e', Customer_Name='Osha Tella', Gender='Male', Age=60, State='Kerala', City='Thiruvananthapuram', Bank_Branch='Thiruvananthapuram Branch', Account_Type='Savings', Transaction_ID='4fa3208f-9e23-42dc-b330-844829d0c12c', Transaction_Date='23-01-2025', Transaction_Time=datetime.datetime(2026, 7, 3, 16, 4, 7), Transaction_Amount=32415.45, Merchant_ID='214e03c5-5c34-40d1-a66c-f440aa2bbd02', Transaction_Type='Transfer', Merchant_Category='Restaurant', Account_Balance=74557.27, Transaction_Device='Voice Assistant', Transaction_Location='Thiruvananthapuram, Kerala', Device_Type='POS', Is_Fraud=0, Transaction_Currency='INR', Customer_Contact='+9198579XXXXXX', Transaction_Description='Bitcoin transaction', Customer_Email='oshaXXXXX@XXXXX.com')
Row(Customer_ID='7c14ad51-781a-4db9-b7bd-67439c175262', Customer_Name='Hredhaan Khosla', Gender='Female', Age=51, State='Maharashtra', City='Nashik', Bank_Branch='Nashik Branch', Account_Type='Business', T

In [14]:
for index, column in enumerate(df.columns):
    print(index, ":", column)

0 : Customer_ID
1 : Customer_Name
2 : Gender
3 : Age
4 : State
5 : City
6 : Bank_Branch
7 : Account_Type
8 : Transaction_ID
9 : Transaction_Date
10 : Transaction_Time
11 : Transaction_Amount
12 : Merchant_ID
13 : Transaction_Type
14 : Merchant_Category
15 : Account_Balance
16 : Transaction_Device
17 : Transaction_Location
18 : Device_Type
19 : Is_Fraud
20 : Transaction_Currency
21 : Customer_Contact
22 : Transaction_Description
23 : Customer_Email


In [15]:
mapped_rdd = transaction_rdd.map(
    lambda x: (x["Customer_ID"], x["Transaction_Amount"])
)

mapped_rdd.take(10)

[('d5f6ec07-d69e-4f47-b9b4-7c58ff17c19e', 32415.45),
 ('7c14ad51-781a-4db9-b7bd-67439c175262', 43622.6),
 ('3a73a0e5-d4da-45aa-85f3-528413900a35', 63062.56),
 ('7902f4ef-9050-4a79-857d-9c2ea3181940', 14000.72),
 ('3a4bba70-d9a9-4c5f-8b92-1735fd8c19e9', 18335.16),
 ('6c870d65-76b0-431d-bdf3-9292998e8211', 9711.15),
 ('5323737c-bbd2-423f-9c9b-e0433c8f75dc', 94677.01),
 ('c0c3d474-f6c2-4c66-9b0e-f9ba75c6f310', 67704.28),
 ('e9a82764-1253-4a46-ad34-80e3416fc801', 72953.45),
 ('708224d5-192a-4d86-b411-8ec6d1bb274b', 5689.02)]

In [16]:
fraud_rdd = transaction_rdd.filter(
    lambda x: x["Is_Fraud"] == 1
)

print("Fraud Transactions :", fraud_rdd.count())

Fraud Transactions : 10088


Q3. Key-Value Operations and Persistence

In [17]:
pair_rdd = transaction_rdd.map(
    lambda x: (x["Customer_ID"], x["Transaction_Amount"])
)

pair_rdd.take(10)

[('d5f6ec07-d69e-4f47-b9b4-7c58ff17c19e', 32415.45),
 ('7c14ad51-781a-4db9-b7bd-67439c175262', 43622.6),
 ('3a73a0e5-d4da-45aa-85f3-528413900a35', 63062.56),
 ('7902f4ef-9050-4a79-857d-9c2ea3181940', 14000.72),
 ('3a4bba70-d9a9-4c5f-8b92-1735fd8c19e9', 18335.16),
 ('6c870d65-76b0-431d-bdf3-9292998e8211', 9711.15),
 ('5323737c-bbd2-423f-9c9b-e0433c8f75dc', 94677.01),
 ('c0c3d474-f6c2-4c66-9b0e-f9ba75c6f310', 67704.28),
 ('e9a82764-1253-4a46-ad34-80e3416fc801', 72953.45),
 ('708224d5-192a-4d86-b411-8ec6d1bb274b', 5689.02)]

In [18]:
# Total Transaction Amount per Customer (reduceByKey)

customer_total = pair_rdd.reduceByKey(lambda x, y: x + y)

customer_total.take(10)

[('3a73a0e5-d4da-45aa-85f3-528413900a35', 63062.56),
 ('3a4bba70-d9a9-4c5f-8b92-1735fd8c19e9', 18335.16),
 ('ef96e796-8b05-47d1-b77d-91651f75e0d4', 78453.41),
 ('f9490190-7155-4551-a5ca-7976abeb82f7', 6552.8),
 ('fcf401c6-f4ca-4e7d-a983-f3fb6d05c1a1', 11598.89),
 ('c5aa88b3-9c56-4486-82bc-b46a8548253b', 24104.53),
 ('75a27a83-361a-480e-a231-f04961930405', 44098.79),
 ('2851a2a2-3d4b-443e-b31b-f13d85d28749', 98260.84),
 ('233c3074-587d-470e-a35a-171efef3f9a3', 2064.89),
 ('4cb49846-1829-4368-950c-2c1170f16adf', 58680.81)]

In [19]:
#Sort by Customer ID

sorted_customers = customer_total.sortByKey()

sorted_customers.take(10)

[('00000d7b-73b6-432c-a706-62447bc30275', 98378.26),
 ('00009fec-323d-4d5d-8919-25be9fb648bd', 23466.43),
 ('000122d9-0442-44a7-9c6a-ec6d8a7135a5', 70357.11),
 ('00014b31-64db-4fa5-af94-96c6c80f6a80', 65838.99),
 ('0001ff91-234e-497a-bd2d-c650c14b8d13', 86337.62),
 ('00024514-bd5f-443d-9ef5-b5e5b2be2433', 92277.42),
 ('00025629-b65d-4ebb-a2f9-aed775c96f0c', 12248.24),
 ('0002a1dc-c0c6-45da-9705-a68f5ddf3965', 1675.26),
 ('0002f886-e348-47b2-89aa-678195939e1c', 18628.6),
 ('00036221-c764-4d6c-9e2c-87ead35ce02b', 68032.97)]

In [20]:
from pyspark import StorageLevel

customer_total.persist(StorageLevel.MEMORY_AND_DISK)

print(customer_total.count())

200000


Q4) Spark DataFrame Operations

In [21]:
df.select(
    "Customer_ID",
    "Transaction_Amount",
    "Transaction_Type"
).show(10)

+--------------------+------------------+----------------+
|         Customer_ID|Transaction_Amount|Transaction_Type|
+--------------------+------------------+----------------+
|d5f6ec07-d69e-4f4...|          32415.45|        Transfer|
|7c14ad51-781a-4db...|           43622.6|    Bill Payment|
|3a73a0e5-d4da-45a...|          63062.56|    Bill Payment|
|7902f4ef-9050-4a7...|          14000.72|           Debit|
|3a4bba70-d9a9-4c5...|          18335.16|        Transfer|
|6c870d65-76b0-431...|           9711.15|        Transfer|
|5323737c-bbd2-423...|          94677.01|        Transfer|
|c0c3d474-f6c2-4c6...|          67704.28|           Debit|
|e9a82764-1253-4a4...|          72953.45|      Withdrawal|
|708224d5-192a-4d8...|           5689.02|          Credit|
+--------------------+------------------+----------------+
only showing top 10 rows


In [22]:
#Filter High Value Transactions  - Filter


high_value = df.filter(
    df.Transaction_Amount > 50000
)

high_value.show(10,truncate=False)

+------------------------------------+---------------+------+---+----------------------------------------+----------+-----------------+------------+------------------------------------+----------------+-------------------+------------------+------------------------------------+----------------+-----------------+---------------+----------------------------+--------------------------------------------------+-----------+--------+--------------------+----------------+--------------------------+------------------------+
|Customer_ID                         |Customer_Name  |Gender|Age|State                                   |City      |Bank_Branch      |Account_Type|Transaction_ID                      |Transaction_Date|Transaction_Time   |Transaction_Amount|Merchant_ID                         |Transaction_Type|Merchant_Category|Account_Balance|Transaction_Device          |Transaction_Location                              |Device_Type|Is_Fraud|Transaction_Currency|Customer_Contact|Transaction

In [23]:
#Branch-wise Total Amount  -  Grouping

from pyspark.sql.functions import sum

branch_amount = df.groupBy(
    "Bank_Branch"
).agg(
    sum("Transaction_Amount").alias("Total_Amount")
)

branch_amount.show()

+------------------+--------------------+
|       Bank_Branch|        Total_Amount|
+------------------+--------------------+
|       Kota Branch|       5.711196641E7|
|   Dehradun Branch| 6.030686223000001E7|
|    Lunglei Branch| 7.016425960000002E7|
|      Ajmer Branch|       5.796764483E7|
|    Jodhpur Branch|        5.73072991E7|
|      Jowai Branch| 7.794341532000001E7|
|       Mahe Branch| 7.281619052999999E7|
|   Diglipur Branch| 9.268339906999998E7|
|       Pune Branch|5.6428776440000005E7|
|     Bhopal Branch| 5.670728520999999E7|
|     Ujjain Branch|5.8543887230000004E7|
|Bhubaneswar Branch|       5.829985913E7|
|  Nongstoin Branch|       7.437874947E7|
|Muzaffarpur Branch| 5.908286875999999E7|
|    Dimapur Branch| 7.270026307999998E7|
|  Mangalore Branch|5.4781758559999995E7|
|   Durgapur Branch|5.9395481800000004E7|
|    Gwalior Branch|5.8558112660000004E7|
|      Patna Branch|       5.548588936E7|
|    Gangtok Branch| 6.963182296000001E7|
+------------------+--------------

In [24]:
customer_total = df.groupBy(
    "Customer_ID"
).agg(
    sum("Transaction_Amount").alias("Total_Amount")
)

joined_df = df.join(
    customer_total,
    on="Customer_ID",
    how="inner"
)

joined_df.select(
    "Customer_ID",
    "Transaction_Amount",
    "Total_Amount"
).show(10)

+--------------------+------------------+------------+
|         Customer_ID|Transaction_Amount|Total_Amount|
+--------------------+------------------+------------+
|32a50238-e369-442...|          80082.34|    80082.34|
|99db16b0-584b-4bf...|          67963.39|    67963.39|
|1ca810ff-c1c1-42b...|          31435.89|    31435.89|
|bfdc780f-845d-46e...|          92998.89|    92998.89|
|7c9049ce-656f-49c...|          40429.36|    40429.36|
|317713a1-1631-4cc...|            654.81|      654.81|
|8867e512-efee-46a...|           79749.5|     79749.5|
|1d55220e-2ea8-41c...|           96581.8|     96581.8|
|b1429f2b-f7d9-4f6...|          54003.67|    54003.67|
|a96438e7-dddf-4c3...|          16794.08|    16794.08|
+--------------------+------------------+------------+
only showing top 10 rows


In [25]:
from pyspark.sql.functions import sum, avg, max, min

df.groupBy(
    "Transaction_Type"
).agg(
    sum("Transaction_Amount").alias("Total"),
    avg("Transaction_Amount").alias("Average"),
    max("Transaction_Amount").alias("Maximum"),
    min("Transaction_Amount").alias("Minimum")
).show()

+----------------+--------------------+------------------+--------+-------+
|Transaction_Type|               Total|           Average| Maximum|Minimum|
+----------------+--------------------+------------------+--------+-------+
|    Bill Payment|1.9796288287000015E9|49441.279438061974|98994.66|  19.48|
|        Transfer|1.9787707423500009E9| 49527.46332816061|98999.98|  10.52|
|      Withdrawal|1.9747900845999987E9|49646.531528270076|98999.02|  12.61|
|          Credit|1.9919708392200005E9| 49576.17817869588|98995.96|  10.29|
|           Debit|1.9824426159699993E9|  49499.1914099875|98999.45|  10.41|
+----------------+--------------------+------------------+--------+-------+



Q5) Exploratory Data Analysis and Spark SQL

In [26]:
df.createOrReplaceTempView("transactions")

In [27]:
# high_value = df.filter(df.Transaction_Amount > 50000)

# high_value.show(10)

In [28]:
spark.sql("""
SELECT *
FROM transactions
WHERE Transaction_Amount > 50000
""").show(10)

+--------------------+---------------+------+---+--------------------+----------+-----------------+------------+--------------------+----------------+-------------------+------------------+--------------------+----------------+-----------------+---------------+--------------------+--------------------+-----------+--------+--------------------+----------------+-----------------------+--------------------+
|         Customer_ID|  Customer_Name|Gender|Age|               State|      City|      Bank_Branch|Account_Type|      Transaction_ID|Transaction_Date|   Transaction_Time|Transaction_Amount|         Merchant_ID|Transaction_Type|Merchant_Category|Account_Balance|  Transaction_Device|Transaction_Location|Device_Type|Is_Fraud|Transaction_Currency|Customer_Contact|Transaction_Description|      Customer_Email|
+--------------------+---------------+------+---+--------------------+----------+-----------------+------------+--------------------+----------------+-------------------+--------------

In [29]:
# import pyspark.sql.functions as F

# customer_pattern = df.groupBy("Customer_ID").agg(
#     F.count("*").alias("Total_Transactions"),
#     F.sum("Transaction_Amount").alias("Total_Amount"),
#     F.avg("Transaction_Amount").alias("Average_Amount")
# )

# customer_pattern.show(10)

In [30]:
spark.sql("""
SELECT
Customer_ID,
COUNT(*) AS Total_Transactions,
SUM(Transaction_Amount) AS Total_Amount,
AVG(Transaction_Amount) AS Average_Amount
FROM transactions
GROUP BY Customer_ID
""").show(10)

+--------------------+------------------+------------+--------------+
|         Customer_ID|Total_Transactions|Total_Amount|Average_Amount|
+--------------------+------------------+------------+--------------+
|32a50238-e369-442...|                 1|    80082.34|      80082.34|
|99db16b0-584b-4bf...|                 1|    67963.39|      67963.39|
|1ca810ff-c1c1-42b...|                 1|    31435.89|      31435.89|
|bfdc780f-845d-46e...|                 1|    92998.89|      92998.89|
|7c9049ce-656f-49c...|                 1|    40429.36|      40429.36|
|317713a1-1631-4cc...|                 1|      654.81|        654.81|
|8867e512-efee-46a...|                 1|     79749.5|       79749.5|
|1d55220e-2ea8-41c...|                 1|     96581.8|       96581.8|
|b1429f2b-f7d9-4f6...|                 1|    54003.67|      54003.67|
|a96438e7-dddf-4c3...|                 1|    16794.08|      16794.08|
+--------------------+------------------+------------+--------------+
only showing top 10 

In [31]:
# branch_volume = df.groupBy("Bank_Branch").agg(
#     F.count("*").alias("Transaction_Count"),
#     F.sum("Transaction_Amount").alias("Total_Amount")
# )

# branch_volume.show()

In [32]:
spark.sql("""
SELECT
Bank_Branch,
COUNT(*) AS Transaction_Count,
SUM(Transaction_Amount) AS Total_Amount
FROM transactions
GROUP BY Bank_Branch
""").show()

+------------------+-----------------+--------------------+
|       Bank_Branch|Transaction_Count|        Total_Amount|
+------------------+-----------------+--------------------+
|       Kota Branch|             1193|       5.711196641E7|
|   Dehradun Branch|             1183| 6.030686223000001E7|
|    Lunglei Branch|             1453| 7.016425960000002E7|
|      Ajmer Branch|             1169|       5.796764483E7|
|    Jodhpur Branch|             1167|        5.73072991E7|
|      Jowai Branch|             1533| 7.794341532000001E7|
|       Mahe Branch|             1481| 7.281619052999999E7|
|   Diglipur Branch|             1926| 9.268339906999998E7|
|       Pune Branch|             1144|5.6428776440000005E7|
|     Bhopal Branch|             1174| 5.670728520999999E7|
|     Ujjain Branch|             1167|5.8543887230000004E7|
|Bhubaneswar Branch|             1186|       5.829985913E7|
|  Nongstoin Branch|             1531|       7.437874947E7|
|Muzaffarpur Branch|             1208| 5

In [33]:
# suspicious = df.filter(
#     (df.Transaction_Amount > 80000) &
#     (df.Is_Fraud == 1)
# )

# suspicious.show()

In [34]:
spark.sql("""
SELECT *
FROM transactions
WHERE
Transaction_Amount > 80000
AND Is_Fraud = 1
""").show()

+--------------------+-----------------+------+---+--------------------+-------------+--------------------+------------+--------------------+----------------+-------------------+------------------+--------------------+----------------+-----------------+---------------+--------------------+--------------------+-----------+--------+--------------------+----------------+-----------------------+--------------------+
|         Customer_ID|    Customer_Name|Gender|Age|               State|         City|         Bank_Branch|Account_Type|      Transaction_ID|Transaction_Date|   Transaction_Time|Transaction_Amount|         Merchant_ID|Transaction_Type|Merchant_Category|Account_Balance|  Transaction_Device|Transaction_Location|Device_Type|Is_Fraud|Transaction_Currency|Customer_Contact|Transaction_Description|      Customer_Email|
+--------------------+-----------------+------+---+--------------------+-------------+--------------------+------------+--------------------+----------------+----------

In [35]:
df.printSchema()

root
 |-- Customer_ID: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- State: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Bank_Branch: string (nullable = true)
 |-- Account_Type: string (nullable = true)
 |-- Transaction_ID: string (nullable = true)
 |-- Transaction_Date: string (nullable = true)
 |-- Transaction_Time: timestamp (nullable = true)
 |-- Transaction_Amount: double (nullable = true)
 |-- Merchant_ID: string (nullable = true)
 |-- Transaction_Type: string (nullable = true)
 |-- Merchant_Category: string (nullable = true)
 |-- Account_Balance: double (nullable = true)
 |-- Transaction_Device: string (nullable = true)
 |-- Transaction_Location: string (nullable = true)
 |-- Device_Type: string (nullable = true)
 |-- Is_Fraud: integer (nullable = true)
 |-- Transaction_Currency: string (nullable = true)
 |-- Customer_Contact: string (nullable = true)
 |-- 

In [36]:
from pyspark.sql.functions import to_date

df = df.withColumn(
    "Transaction_Date",
    to_date("Transaction_Date", "dd-MM-yyyy")
)

Q6. ETL Pipeline Development

In [38]:
# ==========================
# ETL - EXTRACT
# ==========================

# Read the banking transaction dataset

extract_df = spark.read.csv(
    "Bank_Transaction_Fraud_Detection.csv",
    header=True,
    inferSchema=True
)

print("Total Records Extracted:", extract_df.count())

extract_df.show(5)

extract_df.printSchema()

Total Records Extracted: 200000
+--------------------+-------------------+------+---+-----------+------------------+--------------------+------------+--------------------+----------------+-------------------+------------------+--------------------+----------------+-----------------+---------------+------------------+--------------------+-----------+--------+--------------------+----------------+-----------------------+--------------------+
|         Customer_ID|      Customer_Name|Gender|Age|      State|              City|         Bank_Branch|Account_Type|      Transaction_ID|Transaction_Date|   Transaction_Time|Transaction_Amount|         Merchant_ID|Transaction_Type|Merchant_Category|Account_Balance|Transaction_Device|Transaction_Location|Device_Type|Is_Fraud|Transaction_Currency|Customer_Contact|Transaction_Description|      Customer_Email|
+--------------------+-------------------+------+---+-----------+------------------+--------------------+------------+--------------------+-----

In [39]:
# ==========================
# ETL - TRANSFORM
# ==========================

import pyspark.sql.functions as F
from pyspark.sql.functions import to_date, when

transform_df = extract_df

# Remove duplicate records
transform_df = transform_df.dropDuplicates()

# Remove rows with null values in important columns
transform_df = transform_df.dropna(
    subset=[
        "Customer_ID",
        "Transaction_ID",
        "Transaction_Amount",
        "Transaction_Date"
    ]
)

# Convert Transaction_Date to Date format
transform_df = transform_df.withColumn(
    "Transaction_Date",
    to_date("Transaction_Date", "dd-MM-yyyy")
)

# Add Transaction Category
transform_df = transform_df.withColumn(
    "Transaction_Category",
    when(
        F.col("Transaction_Amount") > 50000,
        "High Value"
    ).otherwise("Normal")
)

# Add Fraud Label
transform_df = transform_df.withColumn(
    "Fraud_Label",
    when(
        F.col("Is_Fraud") == 1,
        "Fraud"
    ).otherwise("Legitimate")
)

# Extract Month
transform_df = transform_df.withColumn(
    "Transaction_Month",
    F.month("Transaction_Date")
)

print("Records After Transformation:", transform_df.count())

transform_df.show(10)

Records After Transformation: 200000
+--------------------+----------------+------+---+--------------------+-----------+------------------+------------+--------------------+----------------+-------------------+------------------+--------------------+----------------+-----------------+---------------+--------------------+--------------------+-----------+--------+--------------------+----------------+-----------------------+--------------------+--------------------+-----------+-----------------+
|         Customer_ID|   Customer_Name|Gender|Age|               State|       City|       Bank_Branch|Account_Type|      Transaction_ID|Transaction_Date|   Transaction_Time|Transaction_Amount|         Merchant_ID|Transaction_Type|Merchant_Category|Account_Balance|  Transaction_Device|Transaction_Location|Device_Type|Is_Fraud|Transaction_Currency|Customer_Contact|Transaction_Description|      Customer_Email|Transaction_Category|Fraud_Label|Transaction_Month|
+--------------------+----------------+

In [47]:
import os

# Windows Spark/Hadoop fix: point Spark to a local Hadoop home containing winutils.exe
# If you don't have winutils.exe, download it and place it under C:\hadoop\bin
os.environ['HADOOP_HOME'] = r'C:\hadoop'
os.environ['hadoop.home.dir'] = r'C:\hadoop'
os.environ['PATH'] = os.environ.get('PATH', '') + os.pathsep + os.path.join(os.environ['HADOOP_HOME'], 'bin')

print('HADOOP_HOME set to:', os.environ['HADOOP_HOME'])

HADOOP_HOME set to: C:\hadoop


In [49]:
import os

print(os.path.exists(r"C:\hadoop\bin\winutils.exe"))

False


In [43]:
# # ==========================
# # ETL - LOAD
# # ==========================

# # Save transformed data as CSV
# transform_df.write.mode("overwrite") \
#     .option("header", True) \
#     .csv("etl_csv")

# # Save transformed data as Parquet
# transform_df.write.mode("overwrite") \
#     .parquet("etl_parquet")

# print("ETL Pipeline Executed Successfully!")

# # Verify loaded data
# loaded_df = spark.read.parquet("etl_parquet")

# print("Loaded Records:", loaded_df.count())

# loaded_df.show(5)

In [48]:
# 3. LOAD: Save the processed data into a folder as an optimized Parquet file
output_directory = "F:\\Case_Study_AI_Compute\\etl_cs3.csv"
transform_df.write.mode("overwrite").parquet(output_directory)

Py4JJavaError: An error occurred while calling o421.parquet.
: java.lang.RuntimeException: java.io.FileNotFoundException: java.io.FileNotFoundException: HADOOP_HOME and hadoop.home.dir are unset. -see https://cwiki.apache.org/confluence/display/HADOOP2/WindowsProblems
	at org.apache.hadoop.util.Shell.getWinUtilsPath(Shell.java:789)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:298)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:314)
	at org.apache.hadoop.fs.RawLocalFileSystem.setPermission(RawLocalFileSystem.java:1179)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkOneDirWithMode(RawLocalFileSystem.java:861)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:901)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirs(RawLocalFileSystem.java:873)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:900)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdir

Q7. Machine Learning/Deep Learning Implementation (2 Marks) 
Implement a Machine Learning/Deep Learning model to predict fraudulent transactions or 
customer churn and evaluate its performance.

In [ ]:
# ==========================================
# Q7 - Feature Engineering
# ==========================================

from pyspark.ml.feature import StringIndexer, VectorAssembler

# Convert categorical columns into numeric indices
account_indexer = StringIndexer(
    inputCol="Account_Type",
    outputCol="Account_Type_Index",
    handleInvalid="keep"
)

transaction_indexer = StringIndexer(
    inputCol="Transaction_Type",
    outputCol="Transaction_Type_Index",
    handleInvalid="keep"
)

device_indexer = StringIndexer(
    inputCol="Device_Type",
    outputCol="Device_Type_Index",
    handleInvalid="keep"
)

branch_indexer = StringIndexer(
    inputCol="Bank_Branch",
    outputCol="Bank_Branch_Index",
    handleInvalid="keep"
)

# Fit and transform
df_ml = account_indexer.fit(df).transform(df)
df_ml = transaction_indexer.fit(df_ml).transform(df_ml)
df_ml = device_indexer.fit(df_ml).transform(df_ml)
df_ml = branch_indexer.fit(df_ml).transform(df_ml)

# Combine features into a single vector
assembler = VectorAssembler(
    inputCols=[
        "Age",
        "Transaction_Amount",
        "Account_Balance",
        "Account_Type_Index",
        "Transaction_Type_Index",
        "Device_Type_Index",
        "Bank_Branch_Index"
    ],
    outputCol="features"
)

df_ml = assembler.transform(df_ml)

df_ml.select("features", "Is_Fraud").show(5, truncate=False)

In [ ]:
# ==========================================
# Train Logistic Regression Model
# ==========================================

from pyspark.ml.classification import LogisticRegression

# Split data into training and testing
train_data, test_data = df_ml.randomSplit([0.8, 0.2], seed=42)

# Logistic Regression Model
lr = LogisticRegression(
    featuresCol="features",
    labelCol="Is_Fraud"
)

# Train the model
model = lr.fit(train_data)

print("Model Training Completed")

In [ ]:
# ==========================================
# Prediction and Evaluation
# ==========================================

from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Predict
predictions = model.transform(test_data)

predictions.select(
    "Is_Fraud",
    "prediction",
    "probability"
).show(10, truncate=False)

# Accuracy
accuracy = MulticlassClassificationEvaluator(
    labelCol="Is_Fraud",
    predictionCol="prediction",
    metricName="accuracy"
).evaluate(predictions)

# AUC
auc = BinaryClassificationEvaluator(
    labelCol="Is_Fraud",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
).evaluate(predictions)

print("Accuracy :", accuracy)
print("AUC Score:", auc)